# Comparaison de deux runs — gain point par point

Ce notebook compare **deux runs complets** (typiquement le baseline
`combo_0000` et une config candidate — `auto` (tree), `auto` (RBF), ou
`all_composed`) point par point, et produit directement le tableau et la
figure attendus en section "Gain of the auto strategy relative to the
baseline" du rapport.

Réutilise la même logique que `combo_summary_fixed.ipynb`
(`load_limited_csv`, matching par position plutôt que par valeur brute de
`data_index`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# === CONFIGURATION ===
BASELINE_RESULTS_CSV = Path("results/benchmark/blob_nn_4x10-1/<dossier_combo_0000>/results.csv")
COMPARISON_RESULTS_CSV = Path("results/benchmark/blob_nn_4x10-1/<dossier_recommended_tree>/results.csv")
COMPARISON_LABEL = "auto (tree)"   # utilisé dans les titres/légendes

MIN_RUNS = 78
MAX_RUNS = 78


In [ ]:
def load_limited_csv(csv_path: Path, min_rows: int = MIN_RUNS, max_rows: int = MAX_RUNS) -> pd.DataFrame | None:
    """Charge un CSV, le tronque à max_rows lignes, ou renvoie None s'il en a moins que min_rows."""
    df = pd.read_csv(csv_path)
    n_original = len(df)
    if n_original < min_rows:
        return None
    if n_original > max_rows:
        df = df.iloc[:max_rows].reset_index(drop=True)
    return df


baseline_df = load_limited_csv(BASELINE_RESULTS_CSV)
comparison_df = load_limited_csv(COMPARISON_RESULTS_CSV)

if baseline_df is None:
    raise ValueError(f"Baseline: moins de {MIN_RUNS} lignes utilisables dans {BASELINE_RESULTS_CSV}")
if comparison_df is None:
    raise ValueError(f"Comparaison: moins de {MIN_RUNS} lignes utilisables dans {COMPARISON_RESULTS_CSV}")

print(f"Baseline: {len(baseline_df)} lignes | {COMPARISON_LABEL}: {len(comparison_df)} lignes")


## Matching par position et calcul du gain

In [ ]:
# IMPORTANT: data_index est un compteur qui ne se recoupe pas forcément
# entre deux runs différents (voir combo_summary_fixed.ipynb) — on trie
# chaque fichier par data_index croissant, puis on apparie par position
# relative (1er point testé <-> 1er point testé, etc.), en supposant que
# les deux runs testent les mêmes images dans le même ordre.
baseline_sorted = baseline_df.sort_values("data_index").reset_index(drop=True)
comparison_sorted = comparison_df.sort_values("data_index").reset_index(drop=True)

n_matched = min(len(baseline_sorted), len(comparison_sorted))
if len(baseline_sorted) != len(comparison_sorted):
    print(f"[!] Longueurs différentes ({len(baseline_sorted)} vs {len(comparison_sorted)}), "
          f"on ne compare que les {n_matched} premiers points communs.")

merged = pd.DataFrame({
    "point_rank": range(n_matched),
    "data_index_baseline": baseline_sorted["data_index"].iloc[:n_matched].values,
    "data_index_comparison": comparison_sorted["data_index"].iloc[:n_matched].values,
    "optimal_value_baseline": baseline_sorted["optimal_value"].iloc[:n_matched].values,
    "optimal_value_comparison": comparison_sorted["optimal_value"].iloc[:n_matched].values,
})
merged["certified_baseline"] = merged["optimal_value_baseline"] > 0
merged["certified_comparison"] = merged["optimal_value_comparison"] > 0
merged["gain"] = merged["optimal_value_comparison"] - merged["optimal_value_baseline"]

merged.head()


## Tableau récapitulatif (à coller dans le rapport)

In [ ]:
newly_certified = ((~merged["certified_baseline"]) & (merged["certified_comparison"])).sum()
newly_lost = ((merged["certified_baseline"]) & (~merged["certified_comparison"])).sum()

summary = {
    "Number of matched points": n_matched,
    "Mean gain": merged["gain"].mean(),
    "Median gain": merged["gain"].median(),
    "Points with gain > 0 (%)": (merged["gain"] > 0).mean() * 100,
    "Points with gain < 0 (%)": (merged["gain"] < 0).mean() * 100,
    "Points newly certified (baseline: no -> comparison: yes)": newly_certified,
    "Points newly lost (baseline: yes -> comparison: no)": newly_lost,
    "Certification rate, baseline (%)": merged["certified_baseline"].mean() * 100,
    f"Certification rate, {COMPARISON_LABEL} (%)": merged["certified_comparison"].mean() * 100,
}

summary_df = pd.DataFrame(summary.items(), columns=["Quantity", "Value"])
summary_df


In [ ]:
# Export direct en table LaTeX (\begin{tabular}{lc} ... prêt à coller dans le rapport)
def fmt(v):
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)

latex_rows = "\n".join(
    f"{row.Quantity} & {fmt(row.Value)} \\\\" for row in summary_df.itertuples()
)
latex_table = (
    "\\begin{tabular}{lc}\n\\toprule\n"
    "Quantity & Value \\\\\n\\midrule\n"
    f"{latex_rows}\n"
    "\\bottomrule\n\\end{tabular}"
)
print(latex_table)


## Figure : gain par point, trié par magnitude, statut de certification mis en évidence

In [ ]:
plot_df = merged.sort_values("gain").reset_index(drop=True)
plot_df["x"] = range(len(plot_df))

def flip_category(row):
    if not row["certified_baseline"] and row["certified_comparison"]:
        return "newly certified"
    if row["certified_baseline"] and not row["certified_comparison"]:
        return "newly lost"
    return "unchanged"

plot_df["flip"] = plot_df.apply(flip_category, axis=1)

colors = {"newly certified": "#2a9d8f", "newly lost": "#e76f51", "unchanged": "#888888"}
fig, ax = plt.subplots(figsize=(12, 5))
for flip_type, color in colors.items():
    subset = plot_df[plot_df["flip"] == flip_type]
    ax.scatter(subset["x"], subset["gain"], c=color, s=14,
               alpha=0.5 if flip_type == "unchanged" else 0.9, label=flip_type)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Points de données (triés par gain croissant)")
ax.set_ylabel(f"Gain ({COMPARISON_LABEL} − baseline)")
ax.set_title(f"Gain point par point — {COMPARISON_LABEL} vs baseline")
ax.legend()
plt.tight_layout()

# Police cohérente avec le corps du texte du rapport (taille lisible dans une figure LaTeX)
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(12)
ax.legend(fontsize=11)

plt.savefig(f"auto_vs_baseline_gain__{COMPARISON_LABEL.replace(' ', '_').replace('(', '').replace(')', '')}.pdf",
            dpi=200, bbox_inches="tight")
plt.show()


## Lecture ajustée au coût (nombre de produits en `composed`)

In [ ]:
# Renseignez ces valeurs depuis la section 10 du notebook combo_visualisation
# (n_composed affiché lors de l'écriture du YAML recommandé), et depuis le
# baseline (0 produit composed) et all_composed (330/330).
N_PRODUCTS_TOTAL = 330
N_COMPOSED_IN_COMPARISON = None  # ex: 87

if N_COMPOSED_IN_COMPARISON is not None:
    pct_composed = 100 * N_COMPOSED_IN_COMPARISON / N_PRODUCTS_TOTAL
    print(f"{COMPARISON_LABEL}: {N_COMPOSED_IN_COMPARISON}/{N_PRODUCTS_TOTAL} produits en composed "
          f"({pct_composed:.1f}%)")
    print(f"Gain moyen / % produits composed : {summary['Mean gain'] / pct_composed:.4f}")
else:
    print("Renseignez N_COMPOSED_IN_COMPARISON pour calculer le ratio gain/coût.")
